# Classic ML Preprocessing Workflow
### Copy-paste cheatsheet voor klassiek Machine Learning
**Volgorde:** Data inladen → EDA → Cleaning → Feature Engineering → Encoding → Scaling → Split → Model → Evaluatie

## 1. Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, MinMaxScaler, LabelEncoder
from sklearn.impute import SimpleImputer

# Modellen - classificatie
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import MultinomialNB
from xgboost import XGBClassifier

# Modellen - regressie
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor

# Evaluatie - classificatie
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Evaluatie - regressie
from sklearn.metrics import mean_squared_error, r2_score

## 2. Data Inladen

In [ ]:
# Pas het pad aan naar jouw bestand
df = pd.read_csv('pad/naar/jouw/data.csv')

# Eerste blik
print(df.shape)
print(df.dtypes)
df.head()

## 3. Exploratory Data Analysis (EDA)

In [ ]:
# Basis statistieken
print(df.describe())

# Lege waarden per kolom
print(df.isnull().sum())

# Unieke waarden per kolom (handig voor categorische kolommen)
print(df.nunique())

In [ ]:
# Distributie van numerieke kolommen
df.select_dtypes(include='number').hist(figsize=(15, 10))
plt.tight_layout()
plt.show()

In [ ]:
# Correlatiematrix
corr_matrix = df.select_dtypes(include='number').corr()

plt.figure(figsize=(12, 10))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm')
plt.show()

# Hoge correlaties (> 0.9)
high_corr = corr_matrix.abs() > 0.9
print(corr_matrix[high_corr])

In [ ]:
# Correlatie met target kolom
# Pas 'target' aan naar jouw kolomnaam
print(df.corr(numeric_only=True)['target'].sort_values(ascending=False))

## 4. Data Cleaning

In [ ]:
# Kolommen droppen met meer dan 50% missing values
threshold = len(df) * 0.5
df.dropna(thresh=threshold, axis=1, inplace=True)

# Rijen droppen met missing values
df.dropna(inplace=True)

# OF: missing values opvullen met gemiddelde (numeriek)
df['kolom'] = df['kolom'].fillna(df['kolom'].mean())

# OF: missing values opvullen met meest voorkomende waarde (categorisch)
df['kolom'] = df['kolom'].fillna(df['kolom'].mode()[0])

In [ ]:
# Outliers verwijderen met IQR methode
# Pas 'kolom' aan naar jouw kolomnaam
Q1  = df['kolom'].quantile(0.25)
Q3  = df['kolom'].quantile(0.75)
IQR = Q3 - Q1

df = df[(df['kolom'] > Q1 - 1.5 * IQR) & (df['kolom'] < Q3 + 1.5 * IQR)]

# OF: outliers verwijderen op basis van vaste grens
df = df[df['kolom'] < 500000]

In [ ]:
# Duplicaten verwijderen
df.drop_duplicates(inplace=True)

# Kolommen droppen die niet nuttig zijn
df = df.drop(columns=['kolom1', 'kolom2'])

# Sterk gecorreleerde kolommen droppen
df = df.drop(columns=['gecorreleerde_kolom'])

## 5. Feature Engineering
Nieuwe kolommen berekenen op basis van bestaande kolommen

In [ ]:
# Nieuwe kolommen berekenen via arithmetic
df['rooms_per_household']      = df['total_rooms']    / df['households']
df['population_per_household'] = df['population']     / df['households']
df['bedrooms_per_room']        = df['total_bedrooms'] / df['total_rooms']

# Nieuwe kolom via conditie
df['is_expensive'] = (df['price'] > df['price'].mean()).astype(int)

# Nieuwe categorische kolom via pd.cut
df['price_cat'] = pd.cut(
    df['price'],
    bins=df['price'].quantile([0, 0.25, 0.5, 0.75, 1.0]).values,
    labels=['Low', 'Medium', 'High', 'Very High']
)

## 6. Encoding - Categorische kolommen naar cijfers
**Label Encoding:** kolommen met volgorde (Low < Medium < High)

**One-Hot Encoding:** kolommen zonder volgorde (rood, blauw, groen)

In [ ]:
# One-Hot Encoding (geen volgorde)
# drop_first=True vermijdt dummy trap / multicollineariteit
df = pd.get_dummies(df, columns=['ocean_proximity', 'color'], drop_first=True)

# Label Encoding (wel volgorde: Low=0, Medium=1, High=2)
le = LabelEncoder()
df['size'] = le.fit_transform(df['size'])

# Ordinale mapping (zelf volgorde bepalen)
mapping = {'Low': 0, 'Medium': 1, 'High': 2, 'Very High': 3}
df['price_cat'] = df['price_cat'].map(mapping)

## 7. X en y definiëren

In [ ]:
# Pas 'target' aan naar jouw doelkolom
X = df.drop(columns=['target'])
y = df['target']

print(X.shape)
print(y.value_counts())

## 8. Train/Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

print('Train:', X_train.shape)
print('Test: ', X_test.shape)

## 9. Scaling - Numerieke kolommen normaliseren
**StandardScaler:** mean=0, std=1 → voor LinearRegression, LogisticRegression

**MinMaxScaler:** waarden tussen 0 en 1 → voor Neural Networks

**Niet nodig voor:** RandomForest, XGBoost (beslissingsbomen)

In [ ]:
# StandardScaler
scaler = StandardScaler()

# fit op train, transform op beide
# NOOIT fit op testdata!
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

# OF: MinMaxScaler
# scaler = MinMaxScaler()
# X_train_scaled = scaler.fit_transform(X_train)
# X_test_scaled  = scaler.transform(X_test)

## 10. Model Trainen
### 10a. Classificatie

In [ ]:
# Logistic Regression (eenvoudig, goed startpunt)
lr = LogisticRegression(random_state=42)
lr.fit(X_train_scaled, y_train)

# Random Forest (complexer, vaak beter)
rf = RandomForestClassifier(random_state=42)
rf.fit(X_train, y_train)  # geen scaling nodig

# XGBoost (krachtigst)
xgb = XGBClassifier(random_state=42)
xgb.fit(X_train, y_train)  # geen scaling nodig

### 10b. Regressie

In [ ]:
# Linear Regression
linreg = LinearRegression()
linreg.fit(X_train_scaled, y_train)

# Random Forest Regressor
rfr = RandomForestRegressor(random_state=42)
rfr.fit(X_train, y_train)

# XGBoost Regressor
xgbr = XGBRegressor(random_state=42)
xgbr.fit(X_train, y_train)

## 11. Evaluatie
### 11a. Classificatie

In [ ]:
# Voorspellingen
y_train_pred = lr.predict(X_train_scaled)
y_test_pred  = lr.predict(X_test_scaled)

# Scores
print('TRAIN:')
print('Accuracy:', accuracy_score(y_train, y_train_pred))
print('\nTEST:')
print('Accuracy:', accuracy_score(y_test, y_test_pred))
print(classification_report(y_test, y_test_pred))

# Confusion matrix
sns.heatmap(confusion_matrix(y_test, y_test_pred), annot=True, fmt='d')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.show()

### 11b. Regressie

In [ ]:
# Voorspellingen
y_train_pred = linreg.predict(X_train_scaled)
y_test_pred  = linreg.predict(X_test_scaled)

# Scores
print('TRAIN:')
print('RMSE:', np.sqrt(mean_squared_error(y_train, y_train_pred)))
print('R2:  ', r2_score(y_train, y_train_pred))

print('\nTEST:')
print('RMSE:', np.sqrt(mean_squared_error(y_test, y_test_pred)))
print('R2:  ', r2_score(y_test, y_test_pred))

# Scatter plot voorspellingen vs werkelijke waarden
sns.scatterplot(x=y_test, y=y_test_pred)
plt.xlabel('Werkelijke waarden')
plt.ylabel('Voorspelde waarden')
plt.show()

## 12. Cross Validatie (overfit check)

In [ ]:
# Cross validatie op traindata
scores = cross_val_score(lr, X_train_scaled, y_train, cv=5)
print('CV scores:', scores)
print('Gemiddelde:', scores.mean())

# Als CV score vergelijkbaar is met test score → geen overfit
# Als CV score veel hoger is dan test score → overfit!

## 13. Modellen Vergelijken

In [ ]:
# Classificatie vergelijking
models = {
    'LogisticRegression': LogisticRegression(random_state=42),
    'RandomForest':       RandomForestClassifier(random_state=42),
    'XGBoost':            XGBClassifier(random_state=42)
}

results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    results[name] = accuracy_score(y_test, model.predict(X_test))

for name, score in results.items():
    print(f'{name}: {score:.4f}')